# 멤버 일일 소비 지표 JSON 출력

`data_index_day.ipynb`에서 계산하던 일일 소비 지표를 JSON 직렬화 가능한 단일 결과로 정리합니다.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import cast

import pandas as pd

type JsonScalar = str | int | float | bool | None
type JsonValue = JsonScalar | list[JsonValue] | dict[str, JsonValue]
type JsonObject = dict[str, JsonValue]

In [2]:
def build_daily_consumption_index_json(
    *,
    past_csv_path: str | Path | None = None,
    today_csv_path: str | Path | None = None,
    member_id: int = 1,
    analysis_date: str = "2024-04-01",
    previous_date: str = "2024-03-31",
) -> JsonObject:
    """과거 소비 데이터와 기준일 소비 데이터에서 일일 소비 지표 JSON 객체를 만든다."""
    required_columns = {
        "멤버 id",
        "id",
        "사용 금액",
        "사용 시간",
        "결제 내역",
        "업종 카테고리",
    }
    time_slot_order = {
        "1.새벽(00-06)": 1,
        "2.오전(06-11)": 2,
        "3.점심/오후(11-17)": 3,
        "4.저녁(17-21)": 4,
        "5.밤/야식(21-24)": 5,
    }

    def resolve_path(input_path: str | Path | None, candidates: tuple[Path, ...]) -> Path:
        """직접 받은 경로 또는 실행 위치별 후보 중 실제 존재하는 CSV 경로를 고른다."""
        if input_path is not None:
            return Path(input_path)

        for candidate in candidates:
            if candidate.exists():
                return candidate

        candidate_text = ", ".join(str(candidate) for candidate in candidates)
        raise FileNotFoundError(f"CSV 파일을 찾을 수 없습니다: {candidate_text}")

    def validate_columns(frame: pd.DataFrame, label: str) -> None:
        """입력 CSV에 지표 계산에 필요한 컬럼이 모두 있는지 확인한다."""
        missing_columns = sorted(required_columns.difference(frame.columns))
        if missing_columns:
            raise ValueError(f"{label} 데이터에 필요한 컬럼이 없습니다: {missing_columns}")

    def get_time_slot(hour: int) -> str:
        """결제 시각의 시간을 원본 노트북과 동일한 5개 시간대로 분류한다."""
        if 0 <= hour < 6:
            return "1.새벽(00-06)"
        if 6 <= hour < 11:
            return "2.오전(06-11)"
        if 11 <= hour < 17:
            return "3.점심/오후(11-17)"
        if 17 <= hour < 21:
            return "4.저녁(17-21)"
        return "5.밤/야식(21-24)"

    def safe_rate(numerator: float, denominator: float) -> float:
        """0으로 나누는 상황을 막고 비율 계산 결과를 반환한다."""
        if denominator == 0:
            return 0.0
        return numerator / denominator

    def round_float(value: float, digits: int = 4) -> float:
        """JSON 결과에서 사용하기 쉽게 실수 지표의 소수 자릿수를 제한한다."""
        return round(float(value), digits)

    def to_amount(value: float | int) -> int:
        """원화 합계처럼 정수로 표현할 금액 값을 JSON용 int로 변환한다."""
        return int(round(float(value)))

    def main_category(frame: pd.DataFrame) -> str | None:
        """주어진 거래 데이터에서 사용 금액 합계가 가장 큰 업종 카테고리를 찾는다."""
        if frame.empty:
            return None

        category_totals = frame.groupby("업종 카테고리")["사용 금액"].sum()
        if category_totals.empty:
            return None
        return str(category_totals.idxmax())

    past_path = resolve_path(
        past_csv_path,
        (
            Path("../../../data/raw/csv/transactions_v1.csv"),
            Path("data/raw/csv/transactions_v1.csv"),
        ),
    )
    today_path = resolve_path(
        today_csv_path,
        (
            Path("./data_input_month.csv"),
            Path("notebook/team02/data_pre/data_input_month.csv"),
        ),
    )

    past_frame = pd.read_csv(past_path, encoding="utf-8-sig")
    today_full_frame = pd.read_csv(today_path, encoding="utf-8-sig")
    validate_columns(past_frame, "과거")
    validate_columns(today_full_frame, "기준일")

    analysis_day = pd.to_datetime(analysis_date).date()
    previous_day = pd.to_datetime(previous_date).date()

    past_member_frame = past_frame[past_frame["멤버 id"] == member_id].copy()
    if past_member_frame.empty:
        raise ValueError(f"과거 데이터에서 멤버 {member_id}번 거래를 찾을 수 없습니다.")

    past_member_frame["사용 금액"] = pd.to_numeric(past_member_frame["사용 금액"])
    past_member_frame["사용 시간"] = pd.to_datetime(past_member_frame["사용 시간"])
    past_member_frame["date"] = past_member_frame["사용 시간"].dt.date

    today_full_frame["사용 금액"] = pd.to_numeric(today_full_frame["사용 금액"])
    today_full_frame["사용 시간"] = pd.to_datetime(today_full_frame["사용 시간"])
    today_frame = today_full_frame[
        (today_full_frame["멤버 id"] == member_id)
        & (today_full_frame["사용 시간"].dt.date == analysis_day)
    ].copy()

    q1 = float(past_member_frame["사용 금액"].quantile(0.25))
    q3 = float(past_member_frame["사용 금액"].quantile(0.75))
    iqr = q3 - q1
    lower_bound = max(0.0, q1 - 1.5 * iqr)
    upper_bound = q3 + 1.5 * iqr
    past_member_frame["사용 금액_clipped"] = past_member_frame["사용 금액"].clip(
        lower=lower_bound,
        upper=upper_bound,
    )

    past_daily_stable_avg = float(
        past_member_frame.groupby("date")["사용 금액_clipped"].sum().mean(),
    )
    today_total = float(today_frame["사용 금액"].sum())
    increase_rate = safe_rate(today_total - past_daily_stable_avg, past_daily_stable_avg) * 100

    past_clipped_total = float(past_member_frame["사용 금액_clipped"].sum())
    past_category_ratio = (
        past_member_frame.groupby("업종 카테고리")["사용 금액_clipped"].sum()
        / past_clipped_total
        * 100
    )
    today_category_ratio = (
        today_frame.groupby("업종 카테고리")["사용 금액"].sum() / today_total * 100
        if today_total > 0
        else pd.Series(dtype="float64")
    )
    category_comparison = pd.DataFrame(
        {
            "usual_ratio_percent": past_category_ratio,
            "today_ratio_percent": today_category_ratio,
        },
    ).fillna(0)
    category_comparison["diff_point"] = (
        category_comparison["today_ratio_percent"] - category_comparison["usual_ratio_percent"]
    )

    category_ratio_changes: list[JsonObject] = []
    for category, row in category_comparison.sort_values("diff_point", ascending=False).iterrows():
        category_ratio_changes.append(
            {
                "category": str(category),
                "usual_ratio_percent": round_float(float(row["usual_ratio_percent"])),
                "today_ratio_percent": round_float(float(row["today_ratio_percent"])),
                "diff_point": round_float(float(row["diff_point"])),
            },
        )

    high_spending_frame = today_frame[today_frame["사용 금액"] > upper_bound].copy()
    high_spending_items: list[JsonObject] = []
    for _, row in high_spending_frame.sort_values("사용 금액", ascending=False).iterrows():
        used_at = pd.to_datetime(row["사용 시간"]).strftime("%Y-%m-%d %H:%M:%S")
        high_spending_items.append(
            {
                "used_at": used_at,
                "description": str(row["결제 내역"]),
                "amount": to_amount(row["사용 금액"]),
                "category": str(row["업종 카테고리"]),
            },
        )

    past_daily_original_avg = float(
        past_member_frame.groupby("date")["사용 금액"].sum().mean(),
    )
    spike_ratio = safe_rate(today_total, past_daily_original_avg)

    yesterday_frame = past_member_frame[past_member_frame["date"] == previous_day].copy()
    yesterday_total = float(yesterday_frame["사용 금액"].sum())
    yesterday_count = int(len(yesterday_frame))
    today_count = int(len(today_frame))
    day_diff = today_total - yesterday_total
    day_diff_rate = safe_rate(day_diff, yesterday_total) * 100

    today_frame["hour"] = today_frame["사용 시간"].dt.hour
    today_frame["time_slot"] = today_frame["hour"].apply(get_time_slot)
    past_member_frame["hour"] = past_member_frame["사용 시간"].dt.hour
    past_member_frame["time_slot"] = past_member_frame["hour"].apply(get_time_slot)
    num_past_days = int(past_member_frame["date"].nunique())

    past_time_distribution = (
        past_member_frame.groupby("time_slot")["사용 금액"].sum() / num_past_days
    )
    today_time_distribution = today_frame.groupby("time_slot")["사용 금액"].sum()
    time_comparison = pd.DataFrame(
        {
            "today_amount": today_time_distribution,
            "usual_average_amount": past_time_distribution,
        },
    ).fillna(0)
    time_comparison["sort_order"] = [time_slot_order[str(slot)] for slot in time_comparison.index]
    time_comparison = time_comparison.sort_values("sort_order")

    time_slot_rows: list[JsonObject] = []
    for slot, row in time_comparison.iterrows():
        today_amount = float(row["today_amount"])
        usual_average_amount = float(row["usual_average_amount"])
        time_slot_rows.append(
            {
                "time_slot": str(slot),
                "today_amount": to_amount(today_amount),
                "usual_average_amount": round_float(usual_average_amount),
                "diff_amount": round_float(today_amount - usual_average_amount),
            },
        )

    peak_slot = None if today_time_distribution.empty else str(today_time_distribution.idxmax())

    return {
        "member_id": member_id,
        "analysis_date": str(analysis_day),
        "source_paths": {
            "past_csv": str(past_path),
            "today_csv": str(today_path),
        },
        "outlier_thresholds": {
            "q1": round_float(q1),
            "q3": round_float(q3),
            "iqr": round_float(iqr),
            "lower_bound": round_float(lower_bound),
            "upper_bound": round_float(upper_bound),
        },
        "stable_metrics": {
            "past_daily_stable_average": round_float(past_daily_stable_avg),
            "today_total": to_amount(today_total),
            "increase_rate_percent": round_float(increase_rate),
            "category_ratio_changes": category_ratio_changes,
        },
        "anomaly_detection": {
            "past_daily_original_average": round_float(past_daily_original_avg),
            "spike_ratio": round_float(spike_ratio),
            "is_spike": bool(spike_ratio > 1.5),
            "high_spending_threshold": round_float(upper_bound),
            "high_spending_items": high_spending_items,
        },
        "previous_day_comparison": {
            "yesterday_date": str(previous_day),
            "yesterday_total": to_amount(yesterday_total),
            "today_total": to_amount(today_total),
            "amount_diff": to_amount(day_diff),
            "amount_diff_rate_percent": round_float(day_diff_rate),
            "yesterday_count": yesterday_count,
            "today_count": today_count,
            "count_diff": today_count - yesterday_count,
            "yesterday_main_category": main_category(yesterday_frame),
            "today_main_category": main_category(today_frame),
        },
        "time_slot_analysis": {
            "peak_slot": peak_slot,
            "time_slots": time_slot_rows,
        },
    }

In [3]:
def test_daily_consumption_index_json_contract() -> None:
    """일일 소비 지표 함수가 JSON 직렬화 가능한 핵심 지표를 반환하는지 검증한다."""
    result = build_daily_consumption_index_json()
    stable_metrics = cast(JsonObject, result["stable_metrics"])
    anomaly_detection = cast(JsonObject, result["anomaly_detection"])
    previous_day_comparison = cast(JsonObject, result["previous_day_comparison"])
    time_slot_analysis = cast(JsonObject, result["time_slot_analysis"])
    high_spending_items = cast(list[JsonObject], anomaly_detection["high_spending_items"])

    assert result["member_id"] == 1
    assert result["analysis_date"] == "2024-04-01"
    assert stable_metrics["today_total"] == 133044
    assert high_spending_items[0]["description"] == "SKT통신비"
    assert previous_day_comparison["yesterday_date"] == "2024-03-31"
    assert time_slot_analysis["peak_slot"] == "2.오전(06-11)"

    json.dumps(result, ensure_ascii=False, allow_nan=False)


test_daily_consumption_index_json_contract()

In [4]:
daily_consumption_index_json = build_daily_consumption_index_json()
daily_consumption_index_json_text = json.dumps(
    daily_consumption_index_json,
    ensure_ascii=False,
    indent=2,
    allow_nan=False,
)
print(daily_consumption_index_json_text)

{
  "member_id": 1,
  "analysis_date": "2024-04-01",
  "source_paths": {
    "past_csv": "../../../data/raw/csv/transactions_v1.csv",
    "today_csv": "data_input_month.csv"
  },
  "outlier_thresholds": {
    "q1": 5242.0,
    "q3": 12247.0,
    "iqr": 7005.0,
    "lower_bound": 0.0,
    "upper_bound": 22754.5
  },
  "stable_metrics": {
    "past_daily_stable_average": 51014.0549,
    "today_total": 133044,
    "increase_rate_percent": 160.7987,
    "category_ratio_changes": [
      {
        "category": "생활",
        "usual_ratio_percent": 1.4705,
        "today_ratio_percent": 72.833,
        "diff_point": 71.3626
      },
      {
        "category": "교통",
        "usual_ratio_percent": 4.7418,
        "today_ratio_percent": 9.6885,
        "diff_point": 4.9468
      },
      {
        "category": "의료",
        "usual_ratio_percent": 8.0934,
        "today_ratio_percent": 10.0929,
        "diff_point": 1.9995
      },
      {
        "category": "쇼핑",
        "usual_ratio_percent": 1